# SOLiD-PINN — v0.7.2 One-click CPU reproduction
**Instructions:** Select a CPU runtime, then Runtime → Run all. Authorize Google Drive when asked. If inputs are missing, select the original CSV files (or ZIP containing them) in the upload picker. The notebook verifies all five SHA-256 digests against the historical audits before running. No substitute files, no new fitting, no Model C. Keep Drive private. Never run this notebook concurrently against the same output directory.

Two separate smoke-test invocations verify stage-boundary resume on C/20, then the full 18-stage run starts automatically. Previously completed compatible checkpoints are reused. An interrupted stage restarts. The 11 historical metrics alone cannot restore state. Execution success is not scientific validation.

## 1. Mount Drive and define the fixed run

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, subprocess, hashlib, json, io, zipfile, tempfile

INPUTS_OK = SETUP_OK = SMOKE_OK = False
drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/SOLiD-PINN")
INPUT = ROOT / "private_inputs"
OUTPUT = ROOT / "paper_reproduction_runs"
REPO = Path("/content/SOLiD-PINN")
REV = "aa8a1bb95a444e339d8ecfb45db0661099d2df8f"
ENV = {**os.environ, "OPENBLAS_NUM_THREADS": "1",
       "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1",
       "GIT_TERMINAL_PROMPT": "0"}
EXPECTED = {
    "Ref_data_at_C20.csv": "4b9e0f2688a8db78285f8b7b5ad67db4c28f0479196c2f1c5b3bcdf46a5b4c33",
    "Ref_data_at_C10.csv": "04774e5dc719d75e3f7bd0d3f0447b2b564634e2979ed94451299b143336bb49",
    "Ref_data_at_C5.csv": "bd127f24c6104a70c4756b654b5cf11eab2bc0d7d93708ae7cb8e098326f90ed",
    "SOLiD-PINN-OCV-Figure4/ocv_charge_digitized.csv": "6f4526609a656ee320634d38975c47ea71b36dec08cbfc761d5e4a4469ef4457",
    "SOLiD-PINN-OCV-Figure4/ocv_discharge_digitized.csv": "0ba51db95aca0d9334854251bd14e80104e6fa4afeab34803a99cc5f5bf65d6c"
}
print("Drive mounted. Pinned solver:", REV)

## 2. Verify or upload private inputs
Only upload authorized original files. Existing files with a wrong hash are not overwritten automatically. Remove or relocate an incorrect file yourself and rerun this cell. Uploaded ZIP members are read by basename and hash only; nothing is extracted blindly. The required digests come from the v0.7.2 and COMSOL transfer audits.

In [ ]:
from google.colab import files
INPUTS_OK = SETUP_OK = SMOKE_OK = False
def digest(data):
    return hashlib.sha256(data).hexdigest()

def check_inputs():
    missing, invalid = [], []
    for name, expected in EXPECTED.items():
        p = INPUT / name
        if not p.is_file():
            missing.append(name)
            print("MISSING", name)
        elif digest(p.read_bytes()) != expected:
            invalid.append(name)
            print("HASH MISMATCH", name)
        else:
            print("VERIFIED", name)
    return missing, invalid

missing, invalid = check_inputs()
if invalid:
    raise RuntimeError("Incorrect existing input(s); move them out of private_inputs: " + str(invalid))
if missing:
    print("Select the missing original CSVs or a ZIP containing them. Do not upload to GitHub.")
    old_cwd = Path.cwd()
    with tempfile.TemporaryDirectory() as tmp:
        try:
            os.chdir(tmp)
            uploaded = files.upload()
        finally:
            os.chdir(old_cwd)
        def accept(name, data):
            for rel in missing:
                if Path(name).name == Path(rel).name and digest(data) == EXPECTED[rel]:
                    target = INPUT / rel
                    target.parent.mkdir(parents=True, exist_ok=True)
                    if not target.exists():
                        target.write_bytes(data)
                        print("STORED VERIFIED", rel)
        for name, data in uploaded.items():
            if name.lower().endswith(".zip"):
                with zipfile.ZipFile(io.BytesIO(data)) as z:
                    for member in z.infolist():
                        if not member.is_dir() and any(Path(member.filename).name == Path(rel).name for rel in missing):
                            accept(member.filename, z.read(member))
            else:
                accept(name, data)
        del uploaded
missing, invalid = check_inputs()
if missing or invalid:
    raise RuntimeError("STOP: all five original inputs are required. Missing=" + str(missing) + "; invalid=" + str(invalid))
INPUTS_OK = True
print("INPUTS VERIFIED: 5/5")

## 3. Fetch the fixed revision, install dependencies and run tests
The repository is public at the time this notebook was prepared. No GitHub token is requested. If access changes, stop and use an authorized credential helper. CPU only; keep runtime Python and library versions unchanged on resume.

In [ ]:
assert INPUTS_OK, "Verify all five inputs first."
SETUP_OK = SMOKE_OK = False
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--no-checkout",
                    "https://github.com/theerayutbo/SOLiD-PINN.git", str(REPO)],
                   check=True, env=ENV)
subprocess.run(["git", "fetch", "origin", REV], cwd=REPO, check=True, env=ENV)
subprocess.run(["git", "checkout", "--detach", REV], cwd=REPO, check=True, env=ENV)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == REV
if subprocess.check_output(["git","diff","HEAD","--","scripts","tests","pyproject.toml"], cwd=REPO, text=True).strip():
    raise RuntimeError("Local source modifications detected; use a clean runtime.")
subprocess.run([sys.executable, "-m", "pip", "install",
                "numpy==2.2.6", "scipy==1.15.3", "pandas==2.2.3",
                "-e", ".[dev]"], cwd=REPO, env=ENV, check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q",
                "tests/test_paper_reproduction.py", "tests/test_resumable_reproduction.py"],
               cwd=REPO, env=ENV, check=True)
SETUP_OK = True
print("SETUP AND TESTS PASSED")

## 4. Automatic two-invocation smoke test
The first run computes at most one new C/20 stage. The second run must report charge checkpoint reuse and both selected stages complete. If both already exist, this verifies reuse, not a fresh calculation.

In [ ]:
assert INPUTS_OK and SETUP_OK, "Complete preflight and tests first."
SMOKE_OK = False
CASES = ["literal_0131", "inventory_derived", "historical_kinetics"]
RATES = [20, 10, 5]
def run_job(cases, rates, limit=None):
    cmd = [sys.executable, "-u", "scripts/run_paper_resumable.py",
           str(INPUT), "--outdir", str(OUTPUT)]
    for c in cases:
        cmd += ["--case", c]
    for r in rates:
        cmd += ["--rate", str(r)]
    if limit is not None:
        cmd += ["--limit-new-stages", str(limit)]
    run_dir, status, resumed = None, None, set()
    with subprocess.Popen(cmd, cwd=REPO, env=ENV, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
        for line in p.stdout:
            print(line, end="", flush=True)
            if line.startswith("RUN_DIR "):
                run_dir = Path(line.strip().split(" ", 1)[1])
            elif line.startswith("EXECUTION_STATUS "):
                status = json.loads(line.split(" ", 1)[1])
            elif line.startswith("RESUMED "):
                resumed.add(line.strip().split(" ", 1)[1])
        rc = p.wait()
    if rc != 0 or run_dir is None or status is None or status["status"] == "failed":
        raise RuntimeError(f"Runner failed (exit={rc}, directory={run_dir}); preserve checkpoints and inspect logs.")
    return run_dir, status, resumed

first_dir, first, _ = run_job(["literal_0131"], [20], limit=1)
assert first["completed"] in (1, 2) and first["expected"] == 2
RUN_DIR, second, reused = run_job(["literal_0131"], [20], limit=1)
assert first_dir == RUN_DIR, "Provenance changed between smoke-test invocations."
assert second["status"] == "complete" and second["completed"] == 2 and second["expected"] == 2
assert "literal_0131_20_charge" in reused, "Charge checkpoint reuse not observed."
SMOKE_OK = True
print("SMOKE TEST PASSED: 2/2 stages, charge checkpoint reused.")

## 5. Full run — 18 stages
This cell starts automatically with Run all, but only after the preceding gates pass. Interrupt the cell if needed; rerun from the beginning in the same environment to recover completed checkpoints. Do not run two sessions concurrently.

In [ ]:
assert INPUTS_OK and SETUP_OK and SMOKE_OK, "Smoke test must pass first."
RUN_DIR, final_status, _ = run_job(CASES, RATES)
assert final_status["status"] == "complete" and final_status["completed"] == 18 and final_status["expected"] == 18
print("FULL EXECUTION COMPLETE: 18/18")
print("Private output directory:", RUN_DIR)

## 6. Inspect aggregate metrics and plots
Execution success does not imply the capacity gate passed. Review absolute capacity error ≤3.5%, voltage termination, coverage and voltage RMSE separately. Never share individual traces, raw inputs or concentration states.

In [ ]:
import pandas as pd
from IPython.display import display
if not (RUN_DIR/"metrics.csv").exists():
    raise RuntimeError("No completed stages or recorded failures yet.")
metrics=pd.read_csv(RUN_DIR/"metrics.csv")
display(metrics)
done=metrics[metrics.status.eq("complete")].copy()
expected={(c,r,s) for c in CASES for r in RATES for s in ("charge","discharge")}
observed=set(zip(done.case,done.rate,done.stage))
print(f"Completed: {len(expected & observed)}/{len(expected)}")
print("Remaining:",sorted(expected-observed))
if not done.empty:
    done["capacity_gate"] = done.capacity_error_pct.abs().le(3.5) & done.termination.eq("voltage")
    display(done[["case","rate","stage","termination","capacity_error_pct","rmse_mv","coverage_pct","capacity_gate"]])

In [ ]:
import matplotlib.pyplot as plt
if not done.empty:
    d=done[done.stage.eq("discharge")]
    if not d.empty:
        fig,ax=plt.subplots(figsize=(9,4.5))
        for case,g in d.groupby("case"):
            g=g.sort_values("rate",ascending=False)
            ax.plot(1/g.rate,g.capacity_error_pct,"o-",label=case)
        ax.axhline(3.5,ls="--");ax.axhline(-3.5,ls="--")
        ax.set(xlabel="C-rate (1/h)",ylabel="Signed capacity error (%)",title="Completed discharge stages")
        ax.legend();ax.grid(alpha=.25);fig.tight_layout()
        fig.savefig(RUN_DIR/"capacity_error.png",dpi=180)
        plt.show()

## 7. Handoff
Share only reviewed aggregate metrics, provenance and approved plots after checking confidentiality. Do not start Model C or OFAT before the C/20 configuration selection gate. Source digests are recorded in docs/v072_paper_reproduction_audit.md, docs/comsol_c20_transfer.md and docs/comsol_multirate_holdout.md.